In [ ]:
# ---------------------------------------------
# Synthetic panel for incrementality experiments
# - PREPARATION dataset (pre only, no leakage)
# - VALIDATION dataset (pre + post, lift on treated x post)
# ---------------------------------------------
import numpy as np
import pandas as pd
from datetime import datetime
import json

def make_incrementality_synth(
    pre_start="2024-11-01",
    pre_end="2025-02-28",
    post_end="2025-03-31",
    treated_geos=("NYC","CHI","DAL"),
    n_geos_total=58,
    intervention_date="2025-03-01",
    lift_pct=0.10,
    n_clusters=4,
    seed=42,
    save=True,
    out_prefix="inc"
):
    rng = np.random.default_rng(seed)

    # --- Dates & geos
    pre_start = pd.Timestamp(pre_start)
    pre_end   = pd.Timestamp(pre_end)
    post_end  = pd.Timestamp(post_end)
    t0        = pd.Timestamp(intervention_date)

    assert pre_start < t0 <= post_end, "Date order must be: pre_start < t0 <= post_end"
    dates_all = pd.date_range(pre_start, post_end, freq="D")

    # Build donor list + treated
    donors = [f"GEO_{i:02d}" for i in range(n_geos_total - len(treated_geos))]
    geos = donors + list(treated_geos)

    # --- Latent components
    T = len(dates_all)
    weekly = 10 * np.sin(2*np.pi*np.arange(T)/7.0)             # weekly seasonality
    trend = np.linspace(0, 15, T)                              # mild trend
    # AR(1) global disturbance
    ar = np.zeros(T); phi=0.85; eps = rng.normal(0, 2.0, T)
    for t in range(1, T):
        ar[t] = phi*ar[t-1] + eps[t]

    # Cluster structure
    cluster_ids = rng.integers(0, n_clusters, size=len(geos))
    cluster_map = dict(zip(geos, cluster_ids))
    cluster_factors = {c: (rng.normal(0, 1.0, T).cumsum()/20.0) for c in range(n_clusters)}

    # Geo size multipliers
    size_scale = rng.uniform(0.6, 1.6, size=len(geos))
    size_map = dict(zip(geos, size_scale))

    # --- Generate complete panel (no missing rows)
    rows = []
    baseline = 100.0
    for g in geos:
        c = cluster_map[g]
        sz = size_map[g]
        geo_noise = rng.normal(0, 4.0, T)
        latent = baseline + trend + weekly + ar + 8.0*cluster_factors[c] + geo_noise
        sales = np.maximum(1.0, latent * sz + rng.normal(0, 3.0, T))
        rows.extend((g, dt, float(sales[i])) for i, dt in enumerate(dates_all))

    panel = pd.DataFrame(rows, columns=["geo","date","sales"])

    # --- VALIDATION with treatment flags + lift post-t0
    val = panel.copy()
    val["treated"] = val["geo"].isin(treated_geos).astype(int)
    val["post"] = (val["date"] >= t0).astype(int)
    mask = (val["treated"].eq(1)) & (val["post"].eq(1))
    val.loc[mask, "sales"] *= (1.0 + float(lift_pct))

    # --- PREPARATION (pre only, no treated/post columns)
    prep = panel.loc[panel["date"] < t0, ["geo","date","sales"]].copy()

    # --- Sanity checks (avoid single-geo bugs)
    # 1) Every geo has all pre dates in prep
    expected_pre_days = (t0 - pre_start).days
    counts_prep = prep.groupby("geo")["date"].nunique()
    assert counts_prep.nunique() == 1 and counts_prep.iloc[0] == expected_pre_days, \
        f"Each geo should have {expected_pre_days} pre days; got {counts_prep.describe()}"

    # 2) Every geo has all dates in validation
    expected_all_days = (post_end - pre_start).days + 1
    counts_val = val.groupby("geo")["date"].nunique()
    assert counts_val.nunique() == 1 and counts_val.iloc[0] == expected_all_days, \
        f"Each geo should have {expected_all_days} total days; got {counts_val.describe()}"

    # 3) Treated lift exists post (mean treated_post > treated_pre)
    treated_mask = val["geo"].isin(treated_geos)
    m_pre  = val.loc[treated_mask & (val["post"].eq(0)), "sales"].mean()
    m_post = val.loc[treated_mask & (val["post"].eq(1)), "sales"].mean()
    assert m_post > m_pre, "Expected treated post mean > pre mean (lift not applied?)"

    # --- Save (optional)
    if save:
        prep_path = f"{out_prefix}_prep_df.csv"
        val_path  = f"{out_prefix}_validation_df.csv"
        meta_path = f"{out_prefix}_synth_meta.json"
        prep.to_csv(prep_path, index=False)
        val.to_csv(val_path, index=False)
        meta = {
            "generated_at": datetime.utcnow().isoformat() + "Z",
            "n_geos_total": n_geos_total,
            "treated_geos": list(treated_geos),
            "dates": {"pre_start": str(pre_start.date()), "pre_end": str(pre_end.date()), "post_end": str(post_end.date())},
            "intervention_date": str(t0.date()),
            "lift_pct": float(lift_pct),
            "clusters": int(n_clusters),
            "assertions": {
                "pre_days_per_geo": int(expected_pre_days),
                "total_days_per_geo": int(expected_all_days)
            }
        }
        with open(meta_path, "w") as f:
            json.dump(meta, f, indent=2)
        print(f"Saved: {prep_path}, {val_path}, {meta_path}")

    return prep, val

# ---------- run ----------
prep_df, val_df = make_incrementality_synth()
print(prep_df.shape, val_df.shape)
print("Geos (unique):", prep_df['geo'].nunique(), val_df['geo'].nunique())
print("Dates (prep min/max):", prep_df['date'].min(), prep_df['date'].max())
print("Dates (val  min/max):", val_df['date'].min(),  val_df['date'].max())